# Paper 1 — Original V7 vs. selected V3 inference

**Information-Preserving Adaptive Candidate Selection for Underwater Image Enhancement**

This notebook loads the frozen original V7 baseline and the selected V3 epoch-9 checkpoint, then runs both on exactly the same user image. It displays the real V7/V3 selected outputs, V3's three candidate branches, selector weights, and per-image quality measures. It is for inference and comparison only; it does not train or modify model weights.

The reported V3 validation means use a fixed set of 370 paired validation images and are not measurements of an arbitrary upload. Test-set data is not needed or read here. UIQM/UCIQE are no-reference heuristics, not guarantees of perceived improvement. PSNR/SSIM require a true, aligned reference. Downstream detector/classifier performance is **NOT EVALUATED**. V3 reduced the combined V7 validation objective and improved the four reported image-quality means, while the preservation-loss component regressed. These results are not a uniform improvement in every loss component.

## Architecture and method

The model encodes image condition, creates Conservative/Balanced/Aggressive outputs, scores each through the adaptive selection network using candidate/condition features and fixed preservation/consequence measurements, then returns the candidate selected by the model's own argmax. V3 fine-tuned the existing candidate generator and selector. The condition encoder, information-preservation module, and consequence-check module were frozen. The selected output is not manually replaced with a candidate.

Inference converts EXIF-oriented input to RGB float32 `[0,1]`, preserves dimensions, edge-pads only right/bottom to a multiple of 16 for the model, and crops outputs back to the input size. The same prepared tensor is passed to V7 and V3.

In [ ]:
import os, sys, json, hashlib, subprocess, csv, math, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from IPython.display import display
from google.colab import files

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Selected device:', device)

## Get the verified project source

The original V7 checkpoint is tracked in the repository; this handoff includes the selected V3 checkpoint as an ordinary Git artifact because it is below GitHub's 100 MiB per-file limit. Both are checked against fixed SHA-256 values before loading. No dataset download is required for custom-image inference.

In [ ]:
REPO_URL = 'https://github.com/HariniS1207/Paper1_UIE.git'
ROOT = Path('/content/Paper1_UIE')
if ROOT.exists():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
print('Project:', ROOT)
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())

## Dependencies

The model needs PyTorch; canonical image metrics need NumPy, Pillow, and scikit-image, all listed in `requirements.txt`. This cell keeps the Colab-provided PyTorch build so a compatible GPU/CUDA runtime is not replaced, and installs the project's pinned metric dependencies. Gradio, torchvision, EUVP, and the training environment are unnecessary for this notebook.

In [ ]:
%pip install -q numpy==2.5.2 Pillow==12.3.0 scikit-image==0.26.0
import numpy, skimage
print('NumPy:', numpy.__version__, '| scikit-image:', skimage.__version__, '| torch:', torch.__version__)

## Verify checkpoints and strict-load separate models

Expected files: `checkpoints/v7/paper1_v7_best_epoch_019.pth` (frozen original baseline) and `checkpoints/v7_enhancement_only_v3/paper1_v7_enhancement_only_v3_best_epoch_009.pth` (selected best validation epoch 9 from a 20-epoch run). Strict loading rejects missing or unexpected state keys. The models are placed in evaluation mode.

In [ ]:
CHECKPOINTS = {
 'V7': (ROOT/'checkpoints/v7/paper1_v7_best_epoch_019.pth', 'A0E04FEEC9A6C69CCEE0DD8A6E23E9C35227CDAA10C270DD82F853536920B49E'),
 'V3': (ROOT/'checkpoints/v7_enhancement_only_v3/paper1_v7_enhancement_only_v3_best_epoch_009.pth', '29F99549FBE2ED56ED9B56B4F3F4A34A82A6839AD0678D2F2A4DD91BD7A51EFA'),
}
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest().upper()
for name,(path,expected) in CHECKPOINTS.items():
    actual=sha256(path)
    assert actual==expected, f'{name} SHA-256 mismatch: {actual}'
    print(name, path.relative_to(ROOT), path.stat().st_size, 'bytes', actual)

sys.path.insert(0, str(ROOT/'src'))
from models.paper1_model_v7 import Paper1ModelV7
models={}
for name,(path,_) in CHECKPOINTS.items():
    state=torch.load(path, map_location='cpu', weights_only=False)
    assert 'model_state_dict' in state, f'{name}: model_state_dict missing'
    expected_epoch=19 if name=='V7' else 9
    assert int(state.get('epoch',-1))==expected_epoch, f'{name}: unexpected checkpoint epoch {state.get("epoch")}'
    model=Paper1ModelV7()
    model.load_state_dict(state['model_state_dict'], strict=True)
    model.to(device).eval()
    models[name]=model
print('Both checkpoints strict-loaded successfully.')

## Upload an image and optionally a paired reference

Upload one underwater image. An optional reference is used only for PSNR/SSIM and must be a true paired ground truth, spatially aligned, in the same color space, and exactly the same post-EXIF dimensions. Set the confirmation below to `True` only after verifying those conditions. The notebook will reject a size mismatch rather than resize or register the reference.

In [ ]:
uploaded_input = files.upload()
assert uploaded_input, 'Upload one input image.'
input_path = Path(next(iter(uploaded_input)))
uploaded_reference = files.upload()  # Click Cancel / upload no file to skip reference metrics.
reference_path = Path(next(iter(uploaded_reference))) if uploaded_reference else None
reference_is_paired_aligned_same_color_space = False  # Set True only after checking the conditions above.
if reference_path is not None and not reference_is_paired_aligned_same_color_space:
    raise ValueError('Confirm the optional reference is paired, spatially aligned, and in the same color space before calculating PSNR/SSIM.')
with Image.open(input_path) as im: input_image=ImageOps.exif_transpose(im).convert('RGB').copy()
if reference_path is not None:
    with Image.open(reference_path) as im: reference_image=ImageOps.exif_transpose(im).convert('RGB').copy()
    if reference_image.size != input_image.size:
        raise ValueError(f'Reference size {reference_image.size} differs from input {input_image.size}; no resize/alignment was applied.')
else: reference_image=None
print('Input size:', input_image.size, '| reference:', reference_image.size if reference_image else 'not supplied')

In [ ]:
def prepare_tensor(image):
    arr=np.asarray(image.convert('RGB'), dtype=np.float32)/255.0
    h,w,_=arr.shape
    x=torch.from_numpy(arr.copy()).permute(2,0,1).unsqueeze(0).to(device)
    pad_h=(-h)%16; pad_w=(-w)%16
    if pad_h or pad_w: x=F.pad(x,(0,pad_w,0,pad_h),mode='replicate')
    return x,arr,(w,h),(w+pad_w,h+pad_h)

def output_image(tensor, size):
    w,h=size
    arr=tensor.detach().float().cpu()[0,:,:h,:w].permute(1,2,0).numpy()
    if not np.isfinite(arr).all(): raise ValueError('Model returned non-finite image values.')
    arr=np.clip(arr,0,1)
    return Image.fromarray(np.round(arr*255).astype(np.uint8), mode='RGB')

x,input_rgb,size,padded_size=prepare_tensor(input_image)
with torch.inference_mode():
    v7_output=models['V7'](x)
    v3_output=models['V3'](x)
v7_image=output_image(v7_output['selected_image'],size)
v3_image=output_image(v3_output['selected_image'],size)
v3_candidates={name:output_image(v3_output['candidates'][name],size) for name in ('conservative','balanced','aggressive')}
v3_index=int(v3_output['selected_index'][0].item())
v3_candidate_name=models['V3'].candidate_names[v3_index]
v3_weights=v3_output['selection_weights'][0].detach().float().cpu().tolist()
assert input_image.size==v7_image.size==v3_image.size
print('Original:',size,'| padded model input:',padded_size)
print('V3 actual selected candidate:',v3_candidate_name,'| weights:',dict(zip(models['V3'].candidate_names,v3_weights)))

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,(title,image) in zip(axes,[('Original',input_image),('V7 selected',v7_image),('V3 selected: '+v3_candidate_name,v3_image)]):
    ax.imshow(image); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,(name,image) in zip(axes,v3_candidates.items()):
    ax.imshow(image); ax.set_title('V3 '+name.title()); ax.axis('off')
plt.tight_layout(); plt.show()

## Canonical per-image metrics

Metric function definitions are loaded from `src/evaluate_test.py` using the existing `src/compute_v7_quality_metrics.py` loader. No formulas are copied into the notebook. UIQM/UCIQE are applied to RGB float32 `[0,1]` images. If a reference was supplied, the exact same reference is used for Original, V7, and V3 PSNR/SSIM (`data_range=1.0`, RGB channel axis). These are per-upload values, not validation means.

In [ ]:
from compute_v7_quality_metrics import load_canonical_metric_functions
metric_functions,metric_source_hash=load_canonical_metric_functions(ROOT)
def metric_array(image): return np.asarray(image.convert('RGB'),dtype=np.float32)/255.0
images={'Original':input_image,'V7':v7_image,'V3':v3_image}
rows=[]
ref=metric_array(reference_image) if reference_image is not None else None
for label,image in images.items():
    arr=metric_array(image)
    uiqm=float(metric_functions['uiqm'](np.clip(arr,0,1)))
    uciqe=float(metric_functions['uciqe'](np.clip(arr,0,1)))
    if ref is not None:
        if arr.shape!=ref.shape: raise ValueError(f'{label} dimensions do not match reference.')
        psnr,ssim,_,_=metric_functions['calculate_metrics'](arr,ref)
        psnr=float(psnr); ssim=float(ssim)
    else: psnr=ssim=None
    rows.append({'image':label,'psnr':psnr,'ssim':ssim,'uiqm':uiqm,'uciqe':uciqe})
print('Canonical src/evaluate_test.py SHA-256:',metric_source_hash)
for row in rows: print(row)
print('No reference supplied: PSNR/SSIM are N/A — ground-truth reference required.' if ref is None else 'PSNR/SSIM use the same confirmed aligned reference for all three images.')

## Reproducible weight-difference summary

The project script strictly loads both checkpoints, verifies their hashes, compares parameters and buffers, and writes per-tensor CSV/JSON plus a summary report. It fails if the frozen V3 modules differ from V7.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'src/compare_v7_v3_weights.py')],cwd=ROOT,check=True)
weight_summary=json.loads((ROOT/'results/v7_v3_weight_comparison/weight_comparison.json').read_text())['summary']
print('All-state tensors:',weight_summary['all_state']['tensor_count'])
print('Changed tensors:',weight_summary['all_state']['changed_tensor_count'])
print('Changed elements:',weight_summary['all_state']['changed_elements'])
print('Frozen modules exact:',weight_summary['frozen_modules_exactly_identical'])
display(__import__('pandas').DataFrame([{'module':k,'changed_parameter_tensors':v['parameters']['changed_tensor_count'],'changed_parameter_elements':v['parameters']['changed_elements']} for k,v in weight_summary['modules'].items()]))

In [ ]:
out_dir=ROOT/'colab_outputs'; out_dir.mkdir(exist_ok=True)
for name,image in [('original',input_image),('v7_selected',v7_image),('v3_selected',v3_image),*[(f'v3_{k}',v) for k,v in v3_candidates.items()]]:
    image.save(out_dir/f'{name}.png')
with (out_dir/'image_metrics.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=['image','psnr','ssim','uiqm','uciqe']); writer.writeheader(); writer.writerows(rows)
(out_dir/'weight_comparison_summary.md').write_text((ROOT/'results/v7_v3_weight_comparison/WEIGHT_COMPARISON_REPORT.md').read_text(),encoding='utf-8')
print('Saved outputs and CSV to',out_dir)
with zipfile.ZipFile(out_dir/'comparison_outputs.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in out_dir.iterdir():
        if path.is_file() and path.name!='comparison_outputs.zip': z.write(path,path.name)
files.download(str(out_dir/'comparison_outputs.zip'))

## Reproducibility and limitations

V7 SHA-256: `A0E04FEEC9A6C69CCEE0DD8A6E23E9C35227CDAA10C270DD82F853536920B49E`. V3 epoch-9 SHA-256: `29F99549FBE2ED56ED9B56B4F3F4A34A82A6839AD0678D2F2A4DD91BD7A51EFA`. Outputs are evaluated with both frozen checkpoints in `eval()` and `torch.inference_mode()` on the same input and preprocessing. No training, test split, or checkpoint modifications occur.

PSNR/SSIM are only meaningful when the reference is a valid paired and aligned target. UIQM/UCIQE can react to color cast, contrast, and sharpness in ways that may not agree with human judgment. The reported preservation-loss regression deserves follow-up. Detection loss, classification loss, and classification accuracy were not evaluated; visual/quality metrics do not establish downstream task improvement. Runtime hardware and Colab package versions may differ. This notebook has not been executed in Google Colab by this repository author; follow the cells in a fresh Colab runtime and record the runtime details when reproducing.